# INSTRUCTOR GUIDE: Week 3 Session 2 First Real EDA (Titanic)
Not for students. Same cells as the student notebook, with solutions filled in and an **INSTRUCTOR NOTE** after each part explaining what it does, what the chart shows, the expected numbers, and likely questions. Keep `Titanic-Dataset.csv` next to this notebook.

## How to use this guide
You live-code the Your turn solutions. Before class, run this notebook once (Restart, Run All) so you have seen every chart. Read the notes in order: they build the story you will tell.

## Run of show (2 hours)
| Time | Block | What happens |
|---|---|---|
| 0:00 to 0:08 | Check-in | Wins, assignment questions. Restate the mission: the museum wants the story. |
| 0:08 to 0:17 | Section 0 | Rebuild the clean list one step at a time, then the first look at titles. |
| 0:17 to 0:23 | Section 1 | The five-step workflow; numeric vs categorical. |
| 0:23 to 0:52 | Section 2 | Stats, the mean vs median example, histograms, skew, IQR/boxplot, counts. Live-code Your turn 1 to 3. |
| 0:52 to 1:15 | Section 3 | Survival by sex, class, cabin, port, age. crosstab/pivot. Live-code Your turn 4 and 5. |
| 1:15 to 1:25 | Break | |
| 1:25 to 1:47 | Section 4 | Heatmap (the key chart), women and children check, Cherbourg test, age by survival, correlation. Your turn 6 to 8 (skip 7 if short of time). |
| 1:47 to 1:53 | Section 5 | Read the 5 findings and limitations; Your turn 9 caption, one student shares. |
| 1:53 to 2:00 | Section 6 challenge + recap + assignment | If time is short, set the challenge as homework warm-up. |

**If you are running late:** skip Your turn 2, 7 and the challenge. **If early:** let students do Your turn 9 aloud and compare captions.

## Expected numbers cheat sheet
| Item | Value |
|---|---|
| Rows after cleaning | 891, 0 missing |
| Overall survival | 38.4% (342 of 891) |
| Female / male survival | 74.2% / 18.9% |
| Class 1 / 2 / 3 survival | 63.0% / 47.3% / 24.2% |
| Port C / Q / S survival | 55.4% / 39.0% / 33.9% |
| Alone / with family | 30.4% / 50.6% |
| Fare quartiles Q1 to Q4 | 19.7% / 30.4% / 45.5% / 58.1% |
| Women class 1 / 2 / 3 | 96.8% / 92.1% / 50.0% |
| Men class 1 / 2 / 3 | 36.9% / 15.7% / 13.5% |
| Age mean / median | 29.4 / 30 |
| Fare mean / median | 32.20 / 14.45 |
| Fare skew / LogFare skew | 4.79 / about 0.39 |
| Fare IQR fence / outliers | 65.63 / 116 |
| Corr with Survived: IsFemale, Pclass, Fare | 0.54, -0.34, 0.26 |

## Common mistakes
- Averaging Pclass, or treating it as a number in a story ("mean class 2.3").
- Rates without counts: "family size 8 had 0% survival" is 6 people.
- Bar charts of rates without `plt.ylim(0, 1)`, which exaggerates gaps.
- Topic titles ("Fare histogram") instead of takeaway titles.
- Comparing raw histogram counts of survivors vs non-survivors without `density=True`.
- Saying "paying more caused survival" (correlation vs causation).
- Forgetting `plt.show()`, or reusing one figure so charts draw on top of each other.
- `value_counts()` order: use `sort_index()` for ordered categories.
- Running cells out of order: Restart and Run All.

## Likely student questions
- **Why is the mean of Survived the survival rate?** It is the number of 1s divided by the number of people.
- **What is a "good" correlation?** Rough guide: under 0.1 tiny, 0.1 to 0.3 weak, 0.3 to 0.5 moderate, above 0.5 strong. In messy real data, 0.3 is already interesting.
- **Should we remove outliers?** Only if they are errors. High fares here are real.
- **Why not include Sex in the correlation matrix?** It is text; turn it into 0/1 first (Your turn 8).
- **Is 891 enough data?** Enough for clear patterns like sex and class; not enough for tiny groups (seniors, big families).
- **Why did more men die if they were "stronger"?** The lifeboat policy prioritised women and children; this is social, not physical.
- **Can I use seaborn?** Yes, after `pip install seaborn`. It makes some charts shorter, but matplotlib is enough.
- **Does the order of cleaning matter?** Yes: Title must exist before filling Age by title.

# Week 3 Session 2: First Real EDA: The Story the Titanic Data Tells
**Forge Academy | Data Science Foundations**

**By the end of this session you can:**
1. Explain what EDA is and follow a simple workflow: question, summarize, visualize, interpret, caveat.
2. Describe one column at a time with statistics and charts (histograms, bar charts, boxplots).
3. Compare survival across groups with `groupby`, `crosstab` and `pivot_table`.
4. Look at several variables together with a heatmap and a correlation matrix, and know why correlation is not causation.
5. Turn your findings into a short, honest story with clear charts.

**Setup:** keep `Titanic-Dataset.csv` in the same folder as this notebook. Open it in VS Code, select your Python kernel, and run cells from top to bottom. We use pandas and matplotlib. If `import matplotlib` fails, run `pip install matplotlib` in the terminal.

## Today's mission

Last session the Lagos Heritage Museum asked us for **a clean, trustworthy passenger list**. We delivered it.

Now they are back with a bigger request:

**"We have the clean data. Now tell us the story it tells. Who survived, who did not, and why might that be?"**

That is Exploratory Data Analysis (EDA). Every section today answers one museum question:

| Section | Museum question |
|---|---|
| 0 | Can we rebuild the clean list quickly? |
| 1 | How do analysts explore data? |
| 2 | Who was on board? (one column at a time) |
| 3 | Who survived? (survival against one factor) |
| 4 | How do factors combine? |
| 5 | What story do we tell the visitors? |
| 6 | Challenge, recap and assignment |

## 0. Rebuild the clean passenger list

We cleaned the data step by step last session. Here is the same cleaning again, one short step per cell, so we start from trustworthy data:
- Embarked: 2 missing, filled with the most common port.
- Title extracted from Name and rare titles grouped.
- Age: 177 missing, filled with the median age of each title.
- Cabin: 77% missing, kept only as a HasCabin flag, then dropped.
- New columns: FamilySize and Alone.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
raw = pd.read_csv("Titanic-Dataset.csv")
df = raw.copy()

In [ ]:
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

In [ ]:
df["Title"] = df["Name"].str.extract(r", ([A-Za-z ]+)\.")[0].str.strip()
df["Title"] = df["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
df.loc[~df["Title"].isin(["Mr", "Mrs", "Miss", "Master"]), "Title"] = "Rare"

In [ ]:
df["Age"] = df["Age"].fillna(df.groupby("Title")["Age"].transform("median"))

In [ ]:
df["HasCabin"] = df["Cabin"].notnull()
df = df.drop(columns=["Cabin"])

In [ ]:
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["Alone"] = df["FamilySize"] == 1

In [ ]:
df.shape

In [ ]:
df.isnull().sum().sum()

In [ ]:
df.head()

### 0.1 A first look at the new Title column
**Museum question: What kinds of people were on board?**

Titles like Mr, Mrs, Miss and Master tell us sex and rough age in one word. Let us count them.

In [ ]:
df["Title"].value_counts()

Now the median age for each title. This is exactly what we used to fill the missing ages.

In [ ]:
df.groupby("Title")["Age"].median()

Reading it: "Master" was used for boys, so its median age is very low. That is why filling ages by title is better than one overall median.

**INSTRUCTOR NOTE (0.1).** Expected counts: Mr 517, Miss 185, Mrs 126, Master 40, Rare 23. Expected median ages: Master 3.5, Miss 21, Mr 30, Mrs 35, Rare 48.5. Point out that a 4 year old boy filled with the overall median (about 28) would have been a big mistake. Keep this to 3 minutes.

**INSTRUCTOR NOTE (Section 0).** Expected output: `df.shape` gives (891, 15) and the missing count gives 0. Run the cells one at a time and say in one line what each step fixes. If a student gets `FileNotFoundError`, the CSV is not in the notebook's folder. Spend at most 5 minutes here: say "this is last week's cleaning in one place, nothing new". If asked why `[0]` after `str.extract`: extract returns a table, `[0]` takes its first column.

## 1. What is EDA?

**Exploratory Data Analysis** means getting to know a dataset before you build anything or make claims. You look for patterns, surprises and problems.

We follow the same five steps for every question today:

| Step | What you do | Example |
|---|---|---|
| 1. Question | Ask one clear question | "Did women survive more than men?" |
| 2. Summarize | Get the numbers | `groupby("Sex")["Survived"].mean()` |
| 3. Visualize | Draw a chart that shows it | Bar chart of survival rate by sex |
| 4. Interpret | Say what it means in one sentence | "Women survived about 4 times as often." |
| 5. Caveat | Say what could mislead | "This is a pattern, not proof of cause." |

**Two kinds of variables**
- **Numeric** (numbers you can average): Age, Fare, FamilySize. Use histograms and boxplots.
- **Categorical** (labels or groups): Sex, Pclass, Embarked, Title. Use counts and bar charts.

`Pclass` is stored as 1, 2, 3 but it is a category: averaging ticket classes makes no sense.

**INSTRUCTOR NOTE (Section 1).** Key message: EDA is a habit, not a library. The five steps are the structure of the whole class; point back to them in every section. Likely question: "Is EDA the same as data cleaning?" Answer: cleaning makes the data trustworthy; EDA asks what it says. In practice you go back and forth: exploring often reveals new problems to clean.

## 2. Who was on board? One column at a time (univariate analysis)

### 2.1 Describing numbers
**Museum question: How old were the passengers, and what did they pay?**

| Statistic | Meaning |
|---|---|
| mean | the average; pulled by extreme values |
| median | the middle value; not pulled by extremes |
| mode | the most common value |
| std | standard deviation: how spread out values are around the mean |
| quantiles | 25% (Q1), 50% (median), 75% (Q3) cut points |

In [ ]:
df[["Age", "Fare"]].describe().round(2)

In [ ]:
print("Age  mean:", round(df["Age"].mean(), 1), " median:", df["Age"].median(), " mode:", df["Age"].mode()[0])

In [ ]:
print("Fare mean:", round(df["Fare"].mean(), 2), " median:", df["Fare"].median())

**A tiny example to see why.** Five tickets: four ordinary ones and one very expensive one.

In [ ]:
tickets = pd.Series([10, 12, 14, 15, 500])
print("mean:", tickets.mean())
print("median:", tickets.median())

One extreme ticket drags the mean up to 110.2, but the median stays at 14, which looks like a normal ticket. Keep this picture in mind for the real fares below.

**INSTRUCTOR NOTE (tiny example).** Expected: mean 110.2, median 14.0. Ask: "Which number would you tell a new passenger as the typical price?" The median. This sets up the Fare skew and outliers that follow.

Notice: Fare's mean (32.2) is more than double its median (14.45). That tells us a few very expensive tickets are pulling the average up. For Fare, the median is the better "typical" value.

**INSTRUCTOR NOTE (2.1).** Expected: Age mean 29.4, median 30, mode 30, std 13.3, min 0.42, max 80. Fare mean 32.20, median 14.45, std 49.7, min 0, max 512.33. Explain std simply: "on average, ages sit about 13 years away from the mean". The mode of Age is 30 partly because we filled many missing Mr ages with 30 (the Mr median). Good honest caveat to mention: imputation creates a spike.

### 2.2 Histograms: the shape of a numeric column
A **histogram** splits values into ranges (bins) and counts how many fall in each. Tall bars = many passengers in that range.

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df["Age"], bins=30, color="steelblue", edgecolor="white")
plt.title("Most passengers were young adults (20 to 40)")
plt.xlabel("Age (years)")
plt.ylabel("Number of passengers")
plt.show()

Every chart has a **title that states the takeaway**, and labelled axes. We will keep that habit all session.

### Your turn 1
Draw a histogram of **Fare** with 40 bins. Give it a takeaway title and axis labels. What shape do you see?

In [ ]:
# SOLUTION (live-code this)
plt.figure(figsize=(8, 4))
plt.hist(df["Fare"], bins=40, color="darkorange", edgecolor="white")
plt.title("Most fares were low; a few were very high")
plt.xlabel("Fare (pounds)")
plt.ylabel("Number of passengers")
plt.show()

**INSTRUCTOR NOTE (Age and Fare histograms).** Age chart: one main hump between 20 and 40, a small bump of babies and young children near 0 to 5, few people over 60. The very tall bar around 28 to 32 is partly our imputation (many filled ages). Fare chart: a huge pile on the left (under 30) and a long thin tail to the right up to 512. This is called **right skew**. Say: "most people paid little, a few paid a fortune".

### 2.3 Skew and the log trick
**Skew** measures how lopsided a distribution is. 0 = balanced. Above 1 = a long tail to the right.

A common fix for heavy right skew is to look at the **log** of the values. Log squashes big numbers more than small ones, so the shape becomes easier to see. We use `np.log1p`, which is log(1 + x), so a fare of 0 does not break it.

In [ ]:
import numpy as np

In [ ]:
print("Age skew:", round(df["Age"].skew(), 2))

In [ ]:
print("Fare skew:", round(df["Fare"].skew(), 2))

In [ ]:
df["LogFare"] = np.log1p(df["Fare"])
print("LogFare skew:", round(df["LogFare"].skew(), 2))

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df["LogFare"], bins=30, color="seagreen", edgecolor="white")
plt.title("On a log scale, fares look much more balanced")
plt.xlabel("log(1 + Fare)")
plt.ylabel("Number of passengers")
plt.show()

**INSTRUCTOR NOTE (skew).** Expected: Age skew 0.44 (fairly balanced), Fare skew 4.79 (very skewed), LogFare skew about 0.39. You do not need the maths: "log turns multiplying into adding, so 10, 100, 1000 become evenly spaced". Likely question: "Do we change the real fares?" No, LogFare is a new column for viewing or modelling; the original Fare stays. Second likely question: "Why log1p and not log?" log(0) is undefined and 15 passengers paid 0.

### 2.4 Outliers: IQR and boxplots
**Museum question: Were some fares unusually high?**

The **IQR** (interquartile range) is Q3 minus Q1: the range of the middle 50% of values. A common rule: a value is an **outlier** if it is above Q3 + 1.5 x IQR or below Q1 - 1.5 x IQR.

In [ ]:
q1 = df["Fare"].quantile(0.25)
q3 = df["Fare"].quantile(0.75)
print("Q1:", round(q1, 2), " Q3:", round(q3, 2))

In [ ]:
iqr = q3 - q1
upper = q3 + 1.5 * iqr
print("IQR:", round(iqr, 2), " Upper fence:", round(upper, 2))

In [ ]:
print("Fares above the fence:", (df["Fare"] > upper).sum())

In [ ]:
plt.figure(figsize=(8, 3))
plt.boxplot(df["Fare"], vert=False)
plt.title("Fare: a box of typical fares and a long line of high outliers")
plt.xlabel("Fare (pounds)")
plt.show()

**How to read a boxplot:** the box spans Q1 to Q3, the line inside is the median, the whiskers reach the furthest normal values, and dots are outliers.

**Outlier does not mean error.** These high fares are real first class tickets. We keep them, but we remember they pull the mean up.

**INSTRUCTOR NOTE (IQR and boxplot).** Expected: Q1 7.91, Q3 31.00, IQR 23.09, upper fence 65.63, 116 fares above it. Lower fence is negative, so no low outliers. On the chart: the box is squashed on the left (7.9 to 31), the median line is near 14, and a long trail of dots runs out to 512. Key teaching point: 116 "outliers" is 13% of passengers, so in skewed data the 1.5 x IQR rule flags many real values. Do not delete outliers without a reason.

### Your turn 2
Compute Q1, Q3, IQR and the upper fence for **Age**. How many ages are outliers? Then draw a boxplot of Age.

In [ ]:
# SOLUTION (live-code this)
a1 = df["Age"].quantile(0.25)
a3 = df["Age"].quantile(0.75)
a_iqr = a3 - a1
a_upper = a3 + 1.5 * a_iqr
a_lower = a1 - 1.5 * a_iqr
print("Q1:", a1, " Q3:", a3, " IQR:", a_iqr, " fences:", a_lower, a_upper)

In [ ]:
print("Age outliers:", ((df["Age"] > a_upper) | (df["Age"] < a_lower)).sum())

In [ ]:
plt.figure(figsize=(8, 3))
plt.boxplot(df["Age"], vert=False)
plt.title("Age: a few very old and very young passengers stand out")
plt.xlabel("Age (years)")
plt.show()

**INSTRUCTOR NOTE (Your turn 2).** Because the filled ages bunch near 30, the box is narrow: Q1 21, Q3 35, IQR 14, fences 0 and 56. 35 ages are outliers (everyone over 56). Point out: our imputation made the box narrower than the real age spread, another honest caveat.

### 2.5 Counting categories
**Museum question: Who travelled? How many men and women, in which class, from which port?**

For categories we count with `value_counts()` and draw a **bar chart**. pandas can plot directly with `.plot(kind="bar")`.

In [ ]:
df["Sex"].value_counts()

In [ ]:
df["Sex"].value_counts(normalize=True).round(3)

In [ ]:
df["Sex"].value_counts().plot(kind="bar", color=["steelblue", "salmon"], figsize=(5, 4))
plt.title("Almost two thirds of passengers were men")
plt.xlabel("Sex")
plt.ylabel("Number of passengers")
plt.xticks(rotation=0)
plt.show()

### Your turn 3
Make count bar charts for **Pclass** and **Embarked**. Use `sort_index()` for Pclass so classes appear in order 1, 2, 3. Add takeaway titles.

In [ ]:
# SOLUTION (live-code this)
df["Pclass"].value_counts().sort_index().plot(kind="bar", color="slategray", figsize=(5, 4))
plt.title("More than half of passengers were in third class")
plt.xlabel("Ticket class")
plt.ylabel("Number of passengers")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# SOLUTION (live-code this)
df["Embarked"].value_counts().plot(kind="bar", color="teal", figsize=(5, 4))
plt.title("Most passengers boarded at Southampton")
plt.xlabel("Port (C = Cherbourg, Q = Queenstown, S = Southampton)")
plt.ylabel("Number of passengers")
plt.xticks(rotation=0)
plt.show()

**INSTRUCTOR NOTE (counts).** Expected: male 577 (64.8%), female 314 (35.2%). Pclass: 3rd 491 (55%), 1st 216, 2nd 184. Embarked: S 646, C 168, Q 77. Why this matters for the story: any overall survival rate is dominated by third class men, simply because there were so many of them. Common mistake: forgetting `sort_index()`, so bars appear 3, 1, 2 (value_counts sorts by count).

## 3. Who survived? Survival against one factor (bivariate analysis)

Because `Survived` is 0 or 1, **its mean is the survival rate**. Example: values 1, 0, 0, 1 have mean 0.5, which is 50% survived.

In [ ]:
print("Overall survival rate:", round(df["Survived"].mean() * 100, 1), "%")

In [ ]:
df["Survived"].value_counts()

**INSTRUCTOR NOTE.** Expected: 38.4% survived (342 of 891), 549 died. This is the baseline. Every group rate should be compared with 38.4%: above it, the group did better than average.

### 3.1 Survival by sex
**Museum question: Did "women and children first" really happen?**

In [ ]:
by_sex = df.groupby("Sex")["Survived"].mean()
by_sex.round(3)

In [ ]:
by_sex.plot(kind="bar", color=["salmon", "steelblue"], figsize=(5, 4))
plt.title("Women survived about 4 times as often as men")
plt.ylabel("Survival rate")
plt.xlabel("Sex")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.show()

**Interpret:** about 74% of women survived against 19% of men. **Caveat:** sex is linked to other things (class, travelling with children), so this alone does not tell us everything.

### 3.2 Survival by class, with `agg`
`agg` gives several numbers at once. Always show the **count** next to a rate: a rate from 7 people is far less reliable than one from 500.

In [ ]:
by_class = df.groupby("Pclass")["Survived"].agg(["count", "sum", "mean"])
by_class.columns = ["passengers", "survivors", "survival_rate"]
by_class.round(3)

In [ ]:
by_class["survival_rate"].plot(kind="bar", color="slategray", figsize=(5, 4))
plt.title("First class passengers were 2.6 times as likely to survive as third")
plt.ylabel("Survival rate")
plt.xlabel("Ticket class")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.show()

### 3.2b Compare with the baseline
A rate means more next to the overall 38.4%. Subtracting the overall rate shows how far above or below average each class was.

In [ ]:
overall = df["Survived"].mean()
(by_class["survival_rate"] - overall).round(3)

Positive means better than average, negative means worse. Third class was about 14 points below average.

### 3.2c Survival by cabin record
**Museum question: Did having a recorded cabin go with survival?** We kept `HasCabin` from last session's cleaning.

In [ ]:
df.groupby("HasCabin")["Survived"].agg(["count", "mean"]).round(3)

In [ ]:
pd.crosstab(df["HasCabin"], df["Pclass"])

**Interpret:** passengers with a recorded cabin survived far more often. **Caveat:** most recorded cabins belong to first class passengers, so this is largely the class story again, not the cabin itself.

**INSTRUCTOR NOTE (3.2b, 3.2c).** Baseline differences: class 1 +0.246, class 2 +0.089, class 3 -0.141. HasCabin: False 687 at 0.300, True 204 at 0.667. Crosstab: of the 204 recorded cabins, 176 are first class, 16 second, 12 third. This is a gentle first look at a confounder before Cherbourg in Section 4.

**INSTRUCTOR NOTE (3.1, 3.2).** Expected: female 0.742, male 0.189. Pclass 1: 216 passengers, 136 survivors, 0.630. Pclass 2: 184, 87, 0.473. Pclass 3: 491, 119, 0.242. Story: a clear staircase down by class. Why? First class cabins were on upper decks near the lifeboats; third class was lower down and further away, and there were not enough lifeboats for everyone. Always set `plt.ylim(0, 1)` for rates so bars are not visually exaggerated.

### 3.3 `crosstab`: counts in a grid
`pd.crosstab` counts how often each combination happens. `normalize="index"` turns each row into percentages.

In [ ]:
pd.crosstab(df["Pclass"], df["Survived"])

In [ ]:
pd.crosstab(df["Pclass"], df["Survived"], normalize="index").round(3)

### 3.4 `pivot_table`: the same idea, more flexible
`pivot_table` groups rows by `index` and applies `aggfunc` (mean by default) to `values`.

In [ ]:
df.pivot_table(index="Embarked", values="Survived", aggfunc=["count", "mean"]).round(3)

**INSTRUCTOR NOTE (crosstab, pivot).** Crosstab Pclass x Survived: class 1 has 80 died / 136 survived, class 2 has 97 / 87, class 3 has 372 / 119. With normalize="index", the "1" column is the survival rate again (0.630, 0.473, 0.242), so students see that groupby-mean, crosstab and pivot_table can give the same answer. Embarked: C 168 at 0.554, Q 77 at 0.390, S 646 at 0.339. Likely question: "Was Cherbourg safer?" No: the port did not cause survival. Cherbourg had a higher share of first class passengers. That is a **confounder**, which we test in Section 4.

### 3.5 Survival by age group
`pd.cut` sorts a number column into labelled ranges (bins).

In [ ]:
bins = [0, 12, 18, 35, 60, 100]
labels = ["Child 0-12", "Teen 13-18", "Young adult 19-35", "Adult 36-60", "Senior 61+"]
df["AgeGroup"] = pd.cut(df["Age"], bins=bins, labels=labels)
df[["Age", "AgeGroup"]].head()

In [ ]:
by_age = df.groupby("AgeGroup", observed=True)["Survived"].agg(["count", "mean"]).round(3)
by_age

In [ ]:
by_age["mean"].plot(kind="bar", color="mediumpurple", figsize=(7, 4))
plt.title("Children had the best survival rate; seniors the worst")
plt.ylabel("Survival rate")
plt.xlabel("Age group")
plt.ylim(0, 1)
plt.xticks(rotation=20)
plt.show()

**INSTRUCTOR NOTE (age groups).** Expected: Child 73 at 0.575, Teen 70 at 0.429, Young adult 530 at 0.353, Adult 196 at 0.398, Senior 22 at 0.227. Children did best, supporting "children first", but not overwhelmingly. Caveat: the senior group is only 22 people, so its rate is shaky. Also, imputed ages pushed many people into "Young adult". `observed=True` just hides empty groups and silences a pandas warning; students do not need to worry about it.

### Your turn 4: family
**Museum question: Did travelling with family help?**
1. Survival rate and count by `Alone`.
2. Survival rate and count by `FamilySize`. Draw a bar chart of the rate.
3. Write one sentence interpreting it, and one caveat.

In [ ]:
# SOLUTION (live-code this)
df.groupby("Alone")["Survived"].agg(["count", "mean"]).round(3)

In [ ]:
# SOLUTION (live-code this)
fam = df.groupby("FamilySize")["Survived"].agg(["count", "mean"]).round(3)
fam

In [ ]:
fam["mean"].plot(kind="bar", color="cadetblue", figsize=(7, 4))
plt.title("Small families (2 to 4) survived best; alone and large families worst")
plt.ylabel("Survival rate")
plt.xlabel("Family size (including the passenger)")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.show()

**Interpretation and caveat:** (write yours here)

**INSTRUCTOR NOTE (Your turn 4).** Expected: Alone 537 at 0.304, with family 354 at 0.506. FamilySize: 1: 0.304, 2: 0.553, 3: 0.578, 4: 0.724, 5: 0.200, 6: 0.136, 7: 0.333, 8 and 11: 0.000. Story: an upside-down U. Small families did best; large families (mostly third class, hard to gather everyone) did worst. Caveat: sizes 5+ have very few people (6 to 22). Also most solo travellers were men, so "alone" is partly "male". Students meet this again in the assignment.

### Your turn 5: fare bands
`pd.qcut(df["Fare"], 4, labels=[...])` splits fares into 4 equal-sized groups (quartiles). Create `FareBand` with labels `"Q1 lowest", "Q2", "Q3", "Q4 highest"`, then find and plot the survival rate per band.

In [ ]:
# SOLUTION (live-code this)
df["FareBand"] = pd.qcut(df["Fare"], 4, labels=["Q1 lowest", "Q2", "Q3", "Q4 highest"])
df["FareBand"].value_counts().sort_index()

In [ ]:
by_fare = df.groupby("FareBand", observed=True)["Survived"].mean().round(3)
by_fare

In [ ]:
by_fare.plot(kind="bar", color="goldenrod", figsize=(6, 4))
plt.title("The more you paid, the more likely you survived")
plt.ylabel("Survival rate")
plt.xlabel("Fare quartile")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.show()

**INSTRUCTOR NOTE (Your turn 5).** Expected: Q1 0.197, Q2 0.304, Q3 0.455, Q4 0.581. A steady climb. `cut` vs `qcut`: cut uses ranges you choose; qcut makes groups with equal numbers of people. Fare and class overlap heavily, so this is mostly the class story again.

## 4. How do factors combine? (multivariate analysis)

### 4.1 Sex and class together
**Museum question: Was it sex or class that mattered? Or both?**

In [ ]:
sex_class = df.pivot_table(index="Sex", columns="Pclass", values="Survived", aggfunc="mean").round(3)
sex_class

A **heatmap** colours each cell of a table by its value. With matplotlib we use `imshow` and write the numbers on top.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
im = ax.imshow(sex_class.values, cmap="RdYlGn", vmin=0, vmax=1)
ax.set_xticks(range(len(sex_class.columns)))
ax.set_xticklabels(["1st", "2nd", "3rd"])
ax.set_yticks(range(len(sex_class.index)))
ax.set_yticklabels(sex_class.index)
for i in range(sex_class.shape[0]):
    for j in range(sex_class.shape[1]):
        ax.text(j, i, round(sex_class.values[i, j], 2), ha="center", va="center")
ax.set_title("Survival rate by sex and class: both mattered")
ax.set_xlabel("Ticket class")
fig.colorbar(im, ax=ax, label="Survival rate")
plt.show()

### 4.1b Women and children first?
We can group by two columns at once. `unstack()` turns the second group into columns so it reads like a table.

In [ ]:
sex_age = df.groupby(["Sex", "AgeGroup"], observed=True)["Survived"].mean().round(3)
sex_age

In [ ]:
sex_age.unstack()

Boys up to 12 survived at a similar rate to girls, but from 13 onwards the male rate drops sharply. That is "women and children first" showing in the data.

**INSTRUCTOR NOTE (4.1b).** Expected female: Child 0.594, Teen 0.750, Young adult 0.751, Adult 0.771, Senior 1.000 (only 3 people). Male: Child 0.561, Teen 0.088, Young adult 0.160, Adult 0.190, Senior 0.105. Key line: a boy of 12 had about a 56% chance, a boy of 14 under 10%. Caveat: group sizes vary (female seniors 3), and ages were partly imputed by title.

**INSTRUCTOR NOTE (4.1, the most important chart today).** Expected: female 1st 0.968, 2nd 0.921, 3rd 0.500; male 1st 0.369, 2nd 0.157, 3rd 0.135. How to read it: green = high survival, red = low. Story: almost every first and second class woman survived, but only half of third class women. A first class man (37%) did worse than a third class woman (50%). So sex was the biggest factor, and class made a large difference within each sex. You do not need to explain every line of the heatmap code; the loop just writes each number on its square. Likely question: "Can we use seaborn?" Yes, `sns.heatmap(sex_class, annot=True)` does the same in one line after `pip install seaborn`; we used matplotlib so nothing extra needs installing.

### Your turn 6: test the Cherbourg question
Make a pivot table of survival rate with `Embarked` as rows and `Pclass` as columns. Also make a `crosstab` of Embarked vs Pclass with `normalize="index"` to see the class mix at each port. Was Cherbourg itself lucky, or did it just have more first class passengers?

In [ ]:
# SOLUTION (live-code this)
df.pivot_table(index="Embarked", columns="Pclass", values="Survived").round(3)

In [ ]:
pd.crosstab(df["Embarked"], df["Pclass"], normalize="index").round(3)

**INSTRUCTOR NOTE (Your turn 6).** About half of Cherbourg passengers were first class, against roughly a fifth at Southampton and very few at Queenstown (mostly third class). Within the same class the gap shrinks but does not vanish (1st class: C 0.69 vs S 0.59; 3rd class: C 0.38 vs S 0.19). Conclusion: class mix (a confounder) explains a large part of Cherbourg's advantage; the rest may be other factors (for example more women among Cherbourg passengers), and these groups are small. Honest answer: mostly class, not all. This is the key EDA lesson: a single-factor chart can mislead.

### 4.2 Age by survival
**Museum question: Did age shape survival?** We overlay two histograms: survivors and non-survivors. `alpha` makes them see-through. `density=True` scales each to the same total area so groups of different sizes compare fairly.

In [ ]:
survived = df[df["Survived"] == 1]["Age"]
died = df[df["Survived"] == 0]["Age"]

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(died, bins=30, alpha=0.5, density=True, label="Did not survive", color="firebrick")
plt.hist(survived, bins=30, alpha=0.5, density=True, label="Survived", color="seagreen")
plt.title("Young children survived more; the age shapes are otherwise similar")
plt.xlabel("Age (years)")
plt.ylabel("Share of group")
plt.legend()
plt.show()

In [ ]:
print("Median age, survived:", survived.median(), " did not survive:", died.median())

**INSTRUCTOR NOTE (4.2).** Expected medians: survived 28, did not survive 30. On the chart: green rises above red at ages 0 to about 8 (children were saved), red is higher around 20 to 30 (young men, the largest group), otherwise the shapes overlap a lot. So age matters most at the very young end. Without `density=True`, the red bars would be taller everywhere simply because more people died.

### Your turn 7
Make a boxplot comparing Fare for survivors and non-survivors. Hint: `plt.boxplot([died_fares, survived_fares], labels=[...])`. Use `showfliers=False` to hide outliers so the boxes are readable.

In [ ]:
# SOLUTION (live-code this)
died_fares = df[df["Survived"] == 0]["Fare"]
survived_fares = df[df["Survived"] == 1]["Fare"]

In [ ]:
plt.figure(figsize=(6, 4))
plt.boxplot([died_fares, survived_fares], tick_labels=["Did not survive", "Survived"], showfliers=False)
plt.title("Survivors typically paid higher fares")
plt.ylabel("Fare (pounds)")
plt.show()

In [ ]:
print("Median fare, died:", died_fares.median(), " survived:", survived_fares.median())

**INSTRUCTOR NOTE (Your turn 7).** Expected medians: died 10.5, survived 26.0. The survivors' box sits clearly higher. On older matplotlib versions the argument is `labels=` instead of `tick_labels=`; if a student gets an error, switch it.

### 4.3 Correlation matrix
**Correlation** (from -1 to 1) measures how strongly two numeric columns move together in a straight line.
- Near +1: when one goes up, the other goes up.
- Near -1: when one goes up, the other goes down.
- Near 0: no straight-line relationship (there may still be a curved one).

In [ ]:
num_cols = ["Survived", "Pclass", "Age", "SibSp", "Parch", "Fare", "FamilySize"]
corr = df[num_cols].corr().round(2)
corr

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(num_cols)))
ax.set_xticklabels(num_cols, rotation=45)
ax.set_yticks(range(len(num_cols)))
ax.set_yticklabels(num_cols)
for i in range(len(num_cols)):
    for j in range(len(num_cols)):
        ax.text(j, i, corr.values[i, j], ha="center", va="center", fontsize=8)
ax.set_title("Correlation between numeric columns")
fig.colorbar(im, ax=ax)
plt.show()

**Correlation is not causation.** Fare and survival are positively correlated, but paying more did not *cause* survival. Richer passengers had cabins nearer the lifeboats. A hidden third factor (here, class and cabin location) can create a correlation.

**INSTRUCTOR NOTE (4.3).** Correlation with Survived: Pclass -0.34 (higher class number = lower survival), Fare +0.26, Age -0.08, SibSp -0.04, Parch +0.08, FamilySize +0.02. Pclass and Fare are strongly related to each other (about -0.55). Sex is not in the matrix because it is text; that is a good question to ask the class ("the strongest factor is missing, why?"). FamilySize near 0 does not mean family did not matter: the effect was an upside-down U, which a straight-line measure misses. Diagonal is always 1 (each column with itself).

### Your turn 8
Add a numeric column `IsFemale` (1 for female, 0 for male) using `map`, and find its correlation with `Survived`. How does it compare with Pclass and Fare?

In [ ]:
# SOLUTION (live-code this)
df["IsFemale"] = df["Sex"].map({"female": 1, "male": 0})
print(round(df["IsFemale"].corr(df["Survived"]), 2))

**INSTRUCTOR NOTE (Your turn 8).** Expected: 0.54, the strongest single correlation with survival, bigger than Pclass (-0.34) or Fare (0.26). Confirms the heatmap: sex was the biggest factor.

## 5. Telling the story

**Museum question: What do we put on the exhibition wall?**

Analysis only matters if people understand it. A good finding has: **a number, a comparison, and a plain sentence.**

### Our 5 key findings (from today's numbers)
1. **Most did not survive.** About 38% of the 891 passengers in this data survived.
2. **Sex was the strongest factor.** About 74% of women survived against 19% of men.
3. **Class mattered a lot.** First class 63%, second 47%, third 24%.
4. **Both together tell the real story.** Nearly all first and second class women survived (97% and 92%); third class women had a 50% chance; a first class man (37%) did worse than a third class woman.
5. **Children and small families did better.** Children up to 12 survived at 58%; families of 2 to 4 survived at 55 to 72%, against 30% for people travelling alone.

### Limitations (always say these)
- **Missing ages were estimated.** 177 ages (20%) are title medians, which flattens age patterns.
- **This is not everyone.** The ship carried about 2,224 people; this dataset has 891 passengers (no crew).
- **Survivorship in the records:** information like cabins was better recorded for some groups (mostly first class), so missing data is not random.
- **Patterns, not proof.** We found associations; the causes (deck location, lifeboat rules, crew decisions) come from history, not this table.

### How to communicate
- **Chart titles state the takeaway** ("Women survived 4 times as often"), not just the topic ("Survival by sex").
- **Label axes** with units; start rate axes at 0.
- **One sentence under each chart:** what it shows and why it matters.
- **Show counts** next to rates when groups are small.

### Your turn 9
Pick ONE chart from today and write a museum wall caption for it: a takeaway title, one sentence with a number, and one caveat.

**My caption:** (write yours here)

**INSTRUCTOR NOTE (Section 5).** Read the 5 findings aloud and ask students to point to the chart that proves each one. Example caption for Your turn 9: "Class could decide your fate. 97% of first class women survived, but only 50% of women in third class. Caveat: third class cabins were further from the lifeboats, so location, not ticket price itself, is likely the cause." The "2,224" figure is the commonly cited total aboard (passengers and crew); this Kaggle file is the training portion with 891 passengers.

## 6. Challenge (5 to 10 minutes)
**Museum question: Who were the most and least likely to survive?**

Make a pivot table of survival rate with `Title` as rows and `Pclass` as columns, plus a count table using `aggfunc="count"`. Which title and class combination had the highest survival rate, and which the lowest (only consider groups with at least 10 people)? Write two sentences.

In [ ]:
# SOLUTION (live-code this)
rate = df.pivot_table(index="Title", columns="Pclass", values="Survived", aggfunc="mean").round(3)
rate

In [ ]:
count = df.pivot_table(index="Title", columns="Pclass", values="Survived", aggfunc="count")
count

**INSTRUCTOR NOTE (challenge).** Highest among groups of at least 10: Mrs in 1st class (0.977, 43 people), then Miss 1st (0.958) and Miss 2nd (0.943). Lowest: Mr in 2nd class (0.088, 91 people), then Mr in 3rd (0.113, 319 people). Master (boys) in 1st and 2nd class all survived, but those groups are tiny (3 and 9). Third class boys survived at only 0.393, and third class women at 0.500. Rare in 3rd class is empty (NaN): nobody with a rare title travelled third class. Talking point: titles combine sex and age, which is why they separate survival so well.

## Recap
- EDA follows five steps: question, summarize, visualize, interpret, caveat.
- One column at a time: `describe`, mean vs median, histograms, skew, IQR and boxplots, `value_counts` bar charts.
- Survival against a factor: `groupby(...)["Survived"].agg(["count", "mean"])`, `crosstab`, `pivot_table`, `cut` and `qcut`.
- Factors combine: the Sex x Class heatmap tells a truer story than either alone; correlation is not causation, and confounders (like class behind port) can mislead.
- A finding needs a number, a comparison, a plain sentence and a caveat; chart titles state the takeaway.

## Assignment: your own Titanic EDA
**Due:** before our next class. **Submit:** your `.ipynb` file and your write-up through the usual submission form.

**Task**
1. A clean notebook that starts from `Titanic-Dataset.csv`, cleans it (justify each fix), and explores it with the five-step workflow.
2. **3 to 5 charts**, each with a takeaway title, labelled axes and a one-sentence interpretation. At least one must combine two factors (like Sex x Class).
3. A short **written story** (300 to 600 words): written in a markdown cell at the end of your notebook or a separate document, using your charts, **3 to 5 findings with numbers**, and at least 2 limitations.

**Rubric (20 points, 0 to 4 each)**
| Area | What earns full marks |
|---|---|
| Correctness | Code runs top to bottom; numbers and charts are right |
| Problem solving | Clear questions; goes beyond single-factor charts |
| Code quality | Clear names, no repeated code, raw data untouched |
| Communication | Takeaway titles, labels, readable story, honest caveats |
| Independence | Your own questions and words; you can explain every line |

AI tools are allowed, but you must be able to explain every line and every chart on our call.